# Adult Dataset Preprocessing

This notebook reproduces the preprocessing protocol used for the **Adult** dataset in the GradTree paper:

- Use the 32,561-row `adult.data` file only.
- Treat categorical missing values (?) as an explicit `Unknown` category.
- Create a random 80/20 train-test split, then an 80/20 train-validation split.
- Apply leave-one-out target encoding to all categorical features.
- Apply a normal quantile transform to all features.
- Apply SMOTE to the training split only because the minority class is below 25%.

## 1. Install dependencies

- Colab already includes pandas, NumPy, and scikit-learn. If colab is not used, then installing library using requirements.txt

The two packages below are needed for target encoding and SMOTE. (For colab)

In [1]:
%pip install -q category_encoders imbalanced-learn

^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
import urllib.request
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from category_encoders import LeaveOneOutEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer

SEED = 20770802
QUANTILE_NOISE = 1e-3

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "preprocess":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "dataset" / "adult"
OUTPUT_DIR = PROJECT_ROOT / "preprocess" / "adult_output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Project root: d:\SJSU\Machine Learning\research\gradtree\DecisionTree_GradTree_Reproduction-
Output directory: d:\SJSU\Machine Learning\research\gradtree\DecisionTree_GradTree_Reproduction-\preprocess\adult_output


## 2. Download the source data

- It contains the 32,561 examples reported by the paper. 
- The original UCI URL is tried first, with the GradTree authors' mirror as a fallback.

In [4]:
ADULT_URLS = [
    "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    "https://raw.githubusercontent.com/s-marton/GradTree/master/experiments_paper_gradtree/real_world_datasets/Adult/adult.data",
]
ADULT_PATH = DATA_DIR / "adult.data"

if not ADULT_PATH.exists() or ADULT_PATH.stat().st_size == 0:
    errors = []
    for url in ADULT_URLS:
        try:
            urllib.request.urlretrieve(url, ADULT_PATH)
            break
        except Exception as exc:
            errors.append(f"{url}: {exc}")
    else:
        raise RuntimeError("Could not download adult.data:\n" + "\n".join(errors))

print(f"Source file: {ADULT_PATH}")
print(f"Size: {ADULT_PATH.stat().st_size:,} bytes")

Source file: d:\SJSU\Machine Learning\research\gradtree\DecisionTree_GradTree_Reproduction-\dataset\adult\adult.data
Size: 3,974,305 bytes


## 3. Load and clean the data

The paper keeps all 14 features. The `?` values in `workclass`, `occupation`, and `native-country` are converted to `Unknown` as the paper does not specify an imputation rule.

In [5]:
COLUMNS = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income", # target , all other column are used as features
]
FEATURES = COLUMNS[:-1]
CATEGORICAL_FEATURES = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]
NUMERIC_FEATURES = [column for column in FEATURES if column not in CATEGORICAL_FEATURES]

data = pd.read_csv(
    ADULT_PATH,
    names=COLUMNS,
    skipinitialspace=True,
    na_values=["?"], # make ? to Na 
    skip_blank_lines=True,
)

# strip all the string feature 
for column in data.select_dtypes(include=["object", "string"]).columns:
    data[column] = data[column].str.strip() # strip 

# count the NA cell
missing_counts = {column: int(count) for column, count in data.isna().sum().items()}
# fill the NA with "unknown" for categorical feature 
for column in CATEGORICAL_FEATURES:
    data[column] = data[column].fillna("Unknown")

# Drop rows where target column (income) is missing 
data = data.dropna(subset=["income"]).reset_index(drop=True)

assert len(FEATURES) == 14
assert not data[FEATURES].isna().any().any()

y = data.pop("income").eq(">50K").astype("int8")
X = data[FEATURES].copy()

# we want sorted based on index not value 
target_counts = {int(label): int(count) for label, count in y.value_counts().sort_index().items()}

print(f"Rows: {len(X):,}")
print(f"Features: {X.shape[1]}")
print(f"Target counts (0=<=50K, 1=>50K): {target_counts}")
print(f"Missing values by column before unknown-category filling: {missing_counts}")
display(data.head())

Rows: 32,561
Features: 14
Target counts (0=<=50K, 1=>50K): {0: 24720, 1: 7841}
Missing values by column before unknown-category filling: {'age': 0, 'workclass': 1836, 'fnlwgt': 0, 'education': 0, 'education-num': 0, 'marital-status': 0, 'occupation': 1843, 'relationship': 0, 'race': 0, 'sex': 0, 'capital-gain': 0, 'capital-loss': 0, 'hours-per-week': 0, 'native-country': 583, 'income': 0}


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba


## 4. Create train, validation, and test splits

1. A random 80/20 train-test split 
2. A random 80/20 split of the training portion. Splitting happens before fitting the target encoder or quantile transformer to prevent leakage.

In [6]:
# split dataset -> training set and test set  
X_train_valid, X_test, y_train_valid, y_test = train_test_split( 
    X,
    y,
    test_size=0.20,
    random_state=SEED,
)
# split training set -> train set and valid set 
X_train, X_valid, y_train, y_valid = train_test_split( 
    X_train_valid,
    y_train_valid,
    test_size=0.20,
    random_state=SEED,
)

splits = {
    "train": (X_train, y_train),
    "valid": (X_valid, y_valid),
    "test": (X_test, y_test),
}
for name, (split_X, split_y) in splits.items():
    positive_ratio = float(split_y.mean())
    print(f"{name:5s}: rows={len(split_X):,}, features={split_X.shape[1]}, positive_ratio={positive_ratio:.4f}")

train: rows=20,838, features=14, positive_ratio=0.2412
valid: rows=5,210, features=14, positive_ratio=0.2418
test : rows=6,513, features=14, positive_ratio=0.2388


## 5. Leave-one-out target encoding

The paper uses a specific target encoding method, Leave-one-out, instead of averaging target directly and assign to the corresponding categorical feature to prevent data leakage.

In [7]:
encoder = LeaveOneOutEncoder(
    cols=CATEGORICAL_FEATURES,
    handle_missing="value",
    handle_unknown="value",
    random_state=SEED,
)

X_train_encoded = encoder.fit_transform(X_train, y_train)
# A valid/test row's target was never available to the encoder in the first place !!!
X_valid_encoded = encoder.transform(X_valid)
X_test_encoded = encoder.transform(X_test)

feature_names = list(X_train_encoded.columns)
assert len(feature_names) == 14
assert np.isfinite(X_train_encoded.to_numpy()).all()
assert np.isfinite(X_valid_encoded.to_numpy()).all()
assert np.isfinite(X_test_encoded.to_numpy()).all()

print(f"Encoded feature count: {len(feature_names)}")
display(X_train_encoded.head())

Encoded feature count: 14


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
2232,45,0.219797,339506,0.156598,9,0.043834,0.271259,0.103646,0.124365,0.303951,0,0,40,0.246759
15901,19,0.219797,271118,0.156598,9,0.043834,0.135934,0.010730,0.255837,0.114951,0,0,28,0.246759
30187,33,0.219797,126223,0.156598,9,0.057543,0.121718,0.061003,0.124365,0.114951,0,0,40,0.246759
4643,59,0.219797,381851,0.059375,5,0.074751,0.041589,0.061003,0.124365,0.114951,0,0,40,0.246759
2251,45,0.261614,207847,0.188440,10,0.043834,0.135934,0.061003,0.255837,0.114951,0,0,40,0.246759


## 6. Quantile-normal transformation

A single quantile transformer is fitted to all 14 training features and then reused for validation and test data. Following the authors' implementation, a small seeded Gaussian jitter (`1e-3`) is added only to the copy used for fitting so tied categorical encodings do not create degenerate quantiles.

In [8]:
quantile_fit_data = X_train_encoded.to_numpy(dtype=np.float64, copy=True)
rng = np.random.RandomState(SEED) # for making nose reproducible 
stds = quantile_fit_data.std(axis=0, keepdims=True) # (1, n_features) per column standard deviation 
# Ensure each noise std has a min value so that / noise_std wont be inf 
noise_std = QUANTILE_NOISE / np.maximum(stds, QUANTILE_NOISE)
# Add Gaussian noise 
quantile_fit_data += noise_std * rng.randn(*quantile_fit_data.shape)

quantile_transformer = QuantileTransformer(
    n_quantiles=min(1_000, len(X_train_encoded)),
    output_distribution="normal",
    subsample=None,
    random_state=SEED,
)
# measure distribution based on quantile_fit_data
quantile_transformer.fit(quantile_fit_data)

def apply_quantile_transform(frame: pd.DataFrame) -> pd.DataFrame:
    values = quantile_transformer.transform(frame.to_numpy(dtype=np.float64))
    return pd.DataFrame(values, columns=feature_names, index=frame.index)

X_train_transformed = apply_quantile_transform(X_train_encoded)
X_valid_transformed = apply_quantile_transform(X_valid_encoded)
X_test_transformed = apply_quantile_transform(X_test_encoded)

transformed_stats = X_train_transformed.agg(["mean", "std"]).T
assert X_train_transformed.shape == (len(X_train), 14)
assert np.isfinite(X_train_transformed.to_numpy()).all()
display(transformed_stats)

,mean,std
age,0.000645,0.997311
workclass,0.028305,0.825206
fnlwgt,0.000136,0.999641
education,0.005925,0.954967
education-num,0.003595,0.963512
marital-status,-0.015570,0.752535
occupation,-0.005880,0.940380
relationship,-0.006824,0.886548
race,-0.057386,0.557188
sex,-0.025489,0.653666


## 7. Rebalance the training data

For binary classification, the paper applies SMOTE when the minority fraction is at most `25 / (2 - 1) = 25%`. Adult's minority class is approximately 24.1%, so SMOTE is applied. Validation and test data are never resampled.

In [9]:
num_classes = int(y_train.nunique())
minority_ratio = float(y_train.value_counts(normalize=True).min())
balance_threshold = 0.25 / (num_classes - 1)
smote_applied = minority_ratio <= balance_threshold

if smote_applied:
    X_train_final, y_train_final = SMOTE(random_state=SEED).fit_resample(
        X_train_transformed, # feature based on point A with near point B
        y_train, # target will be the same
    )
    X_train_final = pd.DataFrame(X_train_final, columns=feature_names)
    y_train_final = pd.Series(y_train_final, name="income", dtype="int8")
else:
    X_train_final = X_train_transformed.copy()
    y_train_final = y_train.copy()

assert len(X_valid_transformed) == len(y_valid)
assert len(X_test_transformed) == len(y_test)
assert np.isfinite(X_train_final.to_numpy()).all() # return True if all values are finite 

print(f"Minority ratio before balancing: {minority_ratio:.4f}")
print(f"SMOTE threshold: {balance_threshold:.2f}")
print(f"SMOTE applied: {smote_applied}")
print(f"Training rows after balancing: {len(X_train_final):,}")
print(f"Training target counts after balancing: {y_train_final.value_counts().to_dict()}")

Minority ratio before balancing: 0.2412
SMOTE threshold: 0.25
SMOTE applied: True
Training rows after balancing: 31,624
Training target counts after balancing: {0: 15812, 1: 15812}


## 8. Save processed datasets and fitted transformers

In [10]:
preprocessor_bundle = {
    "encoder": encoder,
    "quantile_transformer": quantile_transformer,
    "feature_names": feature_names,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "random_seed": SEED,
    "quantile_noise": QUANTILE_NOISE,
}
joblib.dump(preprocessor_bundle, OUTPUT_DIR / "adult_preprocessor.joblib")

def save_split(name: str, features: pd.DataFrame, target: pd.Series) -> None:
    output = features.copy()
    output["income"] = np.asarray(target, dtype=np.int8)
    output.to_csv(OUTPUT_DIR / f"adult_{name}.csv", index=False)

save_split("train", X_train_final, y_train_final)
save_split("valid", X_valid_transformed, y_valid)
save_split("test", X_test_transformed, y_test)

summary = {
    "source_rows": int(len(X)),
    "feature_count": int(len(FEATURES)),
    "original_target_counts": target_counts,
    "missing_counts_before_unknown_category": missing_counts,
    "split_rows": {name: int(len(split_X)) for name, (split_X, _) in splits.items()},
    "train_rows_after_smote": int(len(X_train_final)),
    "minority_ratio_before_smote": minority_ratio,
    "smote_threshold": balance_threshold,
    "smote_applied": bool(smote_applied),
    "train_target_counts_after_smote": {
        int(label): int(count) for label, count in y_train_final.value_counts().sort_index().items()
    },
}
(OUTPUT_DIR / "adult_preprocessing_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))
print("Saved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {path.name} ({path.stat().st_size:,} bytes)")

{
  "source_rows": 32561,
  "feature_count": 14,
  "original_target_counts": {
    "0": 24720,
    "1": 7841
  },
  "missing_counts_before_unknown_category": {
    "age": 0,
    "workclass": 1836,
    "fnlwgt": 0,
    "education": 0,
    "education-num": 0,
    "marital-status": 0,
    "occupation": 1843,
    "relationship": 0,
    "race": 0,
    "sex": 0,
    "capital-gain": 0,
    "capital-loss": 0,
    "hours-per-week": 0,
    "native-country": 583,
    "income": 0
  },
  "split_rows": {
    "train": 20838,
    "valid": 5210,
    "test": 6513
  },
  "train_rows_after_smote": 31624,
  "minority_ratio_before_smote": 0.24119397255014877,
  "smote_threshold": 0.25,
  "smote_applied": true,
  "train_target_counts_after_smote": {
    "0": 15812,
    "1": 15812
  }
}
Saved files:
- adult_preprocessing_summary.json (810 bytes)
- adult_preprocessor.joblib (131,051 bytes)
- adult_test.csv (1,843,365 bytes)
- adult_train.csv (8,897,145 bytes)
- adult_valid.csv (1,474,811 bytes)
